# Training-label audit
Read-only checks of runtime-data.csv. No simulations. Run from the challenge root or research directory.

In [1]:
"""Reproduce the proposal's label audit; no simulator or third-party dependencies."""
import csv
import json
import statistics
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'runtime-data.csv').exists() else Path.cwd().parent


def audit():
    rows = list(csv.DictReader((ROOT / 'runtime-data.csv').open()))
    grouped = defaultdict(dict)
    for row in rows:
        grouped[row['filename']][int(row['threshold'])] = row
    result = {
        'rows': len(rows), 'circuits': len(grouped),
        'duplicate_keys': len(rows) - sum(map(len, grouped.values())),
        'settings': {k: sorted({r[k] for r in rows})
                     for k in ('shots', 'backend', 'precision')},
        'threshold_coverage': {str(k): v for k, v in
                               Counter(tuple(sorted(g)) for g in grouped.values()).items()},
        'status_duration_mismatches': sum((r['status'] == 'timeout') !=
                                         (not r['duration_s']) for r in rows),
        'success_above_cap': [r for r in rows if r['status'] == 'success'
                              and float(r['duration_s']) > 14400],
        'by_threshold': {}, 'paired_comparisons': {},
    }
    for threshold in (16, 64, 512):
        subset = [r for r in rows if int(r['threshold']) == threshold]
        durations = [float(r['duration_s']) for r in subset if r['status'] == 'success']
        result['by_threshold'][threshold] = {
            'rows': len(subset), 'status': dict(Counter(r['status'] for r in subset)),
            'success_min': min(durations), 'success_median': statistics.median(durations),
            'success_max': max(durations),
        }
    for low, high in ((16, 64), (64, 512), (16, 512)):
        pairs = [(float(g[low]['duration_s']), float(g[high]['duration_s']))
                 for g in grouped.values() if low in g and high in g
                 and g[low]['status'] == g[high]['status'] == 'success']
        result['paired_comparisons'][f'{low}->{high}'] = {
            'both_success': len(pairs), 'higher_threshold_faster': sum(b < a for a, b in pairs),
            'higher_threshold_at_least_20pct_faster': sum(b < .8*a for a, b in pairs),
            'median_runtime_ratio': statistics.median(b/a for a, b in pairs),
        }
    files = {p.name.removesuffix('.zst') for p in (ROOT / 'training_circuits').glob('*.qasm.zst')}
    result['circuit_files'] = len(files)
    result['missing_files'] = sorted(set(grouped) - files)
    result['unlabeled_files'] = sorted(files - set(grouped))
    return result


if __name__ == '__main__':
    print(json.dumps(audit(), indent=2))


{
  "rows": 1497,
  "circuits": 532,
  "duplicate_keys": 0,
  "settings": {
    "shots": [
      "1024"
    ],
    "backend": [
      "scarlet_quantum_rings"
    ],
    "precision": [
      "single"
    ]
  },
  "threshold_coverage": {
    "(16, 64, 512)": 433,
    "(16, 64)": 97,
    "(16, 512)": 2
  },
  "status_duration_mismatches": 0,
  "success_above_cap": [
    {
      "filename": "2afe3d2e.qasm",
      "shots": "1024",
      "backend": "scarlet_quantum_rings",
      "precision": "single",
      "threshold": "64",
      "duration_s": "42993.9847111",
      "status": "success"
    }
  ],
  "by_threshold": {
    "16": {
      "rows": 532,
      "status": {
        "success": 523,
        "timeout": 9
      },
      "success_min": 0.1633197,
      "success_median": 1.5307986,
      "success_max": 9723.4914956
    },
    "64": {
      "rows": 530,
      "status": {
        "success": 526,
        "timeout": 4
      },
      "success_min": 0.1524136,
      "success_median": 4.15619640